# Exploratory Data Analysis: Raw Data

**Objective**: Analyze the raw irrigation dataset to identify patterns, data quality issues, and feature engineering requirements before model training.

This notebook utilizes ydata-profiling for automated reporting and custom statistical analysis to determine:
- **Data Integrity**: Checking for missingness and duplicates.
- **Distribution Analysis**: Assessing skewness and kurtosis to inform transformation strategies.
- **Encoding Strategies**: Mapping categorical variables for both Tree-based and Linear models.

In [ ]:
# Install required library
!uv pip install --quiet fg-data-profiling

### 1. Environment Setup

In [ ]:
import sys
from pathlib import Path

# Get the path to 'irrigation-prediction-system' (the project root)
root_path: Path = Path.cwd().parent.parent.parent
sys.path.append(str(root_path))

### 2. Import Libraries

In [ ]:
import pandas as pd
from data_profiling import ProfileReport
from prefect.settings import PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW, temporary_settings

from src.configs.data_cfg import FALSE_VALUES, NA_VALUES, OPTIMIZED_DTYPES, TRUE_VALUES
from src.configs.settings import Settings, get_settings
from src.pipeline.ingestion import load_dataset

%matplotlib inline

### 3. Setup Configs

In [ ]:
settings: Settings = get_settings()

# EDA report title
report_title: str = "Compherensive EDA Report: Raw Data"

# Dataset path
raw_data_path: Path = settings.RAW_DATA_DIR / settings.RAW_DATA_FILENAME

### 4. Load the Dataset

In [ ]:
try:
    with temporary_settings(updates={PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW: "ignore"}):
        df = load_dataset(
            raw_data_path,
            sep=",",
            header=0,
            index_col="id",
            true_values=TRUE_VALUES,
            false_values=FALSE_VALUES,
            dtype=OPTIMIZED_DTYPES,
            na_values=NA_VALUES,
            keep_default_na=True,
            on_bad_lines="warn",
            float_precision="round_trip",
            skipinitialspace=True,
            encoding="utf-8",
            encoding_errors="replace",
            memory_map=True,
            low_memory=False,
        )
except Exception as e:
    sys.stderr.write(f"Ingestion failed: {e}\n")
    sys.exit(1)

### 5. Generate EDA Report: Raw Data 

In [ ]:
profile = ProfileReport(df, title=report_title, explorative=True)
profile.to_notebook_iframe()

### 6. More Info and Descriptive Statistics

In [ ]:
cat_cols = df.select_dtypes(include=["category", "boolean"]).columns
num_cols = df.select_dtypes(include=["Float32"]).columns

print("Categorical Columns:", cat_cols)
print("Numerical Columns:", num_cols)

In [ ]:
# Categorical dataframe
cat_df: pd.DataFrame = df[cat_cols].copy()

# Unique values in each categorical column
for col in cat_cols:
    print(f"{col} ({cat_df[col].nunique()}): {cat_df[col].unique().tolist()}")

In [ ]:
# Numerical dataframe
num_df: pd.DataFrame = df[num_cols].copy()

# Statistical info
summary: list = []
for col in num_cols:
    clean = df[col].dropna()
    summary.append(
        {
            "Column": col,
            "Count": len(clean),
            "Mean": clean.mean(),
            "Std": clean.std(),
            "Min": clean.min(),
            "Max": clean.max(),
            "Skewness": clean.skew(),
            "Kurtosis": clean.kurtosis(),
            "Q1": clean.quantile(0.25),
            "Q3": clean.quantile(0.75),
            "IQR": clean.quantile(0.75) - clean.quantile(0.25),
        }
    )

pd.DataFrame(summary).sort_values("Skewness", ascending=False)

### 7. EDA Insights and Action Plan

- There are 20 columns and 6,30,000 rows.
- No missing cells or duplicate rows.
- There are 8 categorical variables, 11 numeric variables, and 1 boolean variable.
- Categorical and boolean variables needs encoding:
    - For tree-based models: Random Forest, XGBoost, LightGBM, CatBoost
        - Ordinal Encoding on `Crop_Growth_Stage`(Sowing → Vegetative → Flowering → Harvest) and `Irrigation_Need`(Low → Medium → High).
        - Integer/Label Encoding or Native Categorical on `Soil_Type`, `Crop_Type`, `Season`, `Irrigation_Type`, `Water_Source` and `Region`.
        - Convert `Mulching_Used` to 0/1.
    - For linear models: Linear/Logistic Regression, SVM, Neural Nets
        - Ordinal Encoding on `Crop_Growth_Stage`(Sowing → Vegetative → Flowering → Harvest) and `Irrigation_Need`(Low → Medium → High).
        - One-Hot Encoding (drop_first=True) on `Soil_Type`, `Crop_Type`, `Season`, `Irrigation_Type`, `Water_Source` and `Region`.
        - Convert `Mulching_Used` to 0/1.
- Skewness for every numerical column is between -0.12 and +0.11, means no Power or Log transforms needed.
- Kurtosis values are all consistently around -0.94 to -1.26, means few extreme outliers presesnt.
- Scaling is needed for linear models using StandardScaler, RobustScaler or MinMaxScaler.
- No scaling for tree models but can cap at 0.5th/99.5th percentile.